# Thresholds - Étape 3

Ce notebook investigue l'optimisation des thresholds

In [1]:
%%html

<!-- styles d'affichage -->
<style>
    .section_div {
        width:70%;
        height:1.5px;
        border:none;
        color:black;
        background-color:black;
        margin: auto;
        margin-top: 0px;
        margin-bottom: 0px;
    }

    .answer {
        color:blue;
    }

    .question {
        color:red;
    }

    .note {
        color:green;
        font-weight: bold;
    }
</style>

In [2]:
# assure le reload de src si modifications sont faite
#
%load_ext autoreload
%autoreload 2

In [3]:
# import utilitaires
#
%matplotlib inline

import librosa as rosa
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from copy import deepcopy
from pathlib import Path
from pprint import pprint
from tqdm.notebook import tqdm

In [4]:
# import package develope pour le projet
#
import ffury

from ffury.configs import (
    DatasetType,
    DEFAULT_CONFIG_FILE,
    load_config
)
from ffury.ml import confusion_matrix_analysis

def get_config():
    # creation config - on sait que DEFAULT_CONFIG_FILE est dans le repertoire parent
    config = Path(ffury.__file__).parents[2].joinpath(DEFAULT_CONFIG_FILE)
    return load_config(config)

config = get_config()

In [5]:
# donnees necessaire pour calculer les thresolds

from ffury.optional.development.dataset import IndexedDataset
from ffury.optional.keras_adapters import _load_model

# loading du model
model = _load_model(config)
model.summary()
print()
model.layers[2].layer.summary()

print()

# loading des donnees
train_data = IndexedDataset.create(config, DatasetType.TRAIN)
x = train_data.spectrogram_groups
y_true = train_data.y
print("x shape:", x.shape)
print("y true shape:", y_true.shape )

print()

# probabilites par le modeles (modele a 2 output ; probabilites + features)
y_pred, _ = model.predict(x, verbose=0)
print("y pred. shape:", y_pred.shape)

2025-01-29 15:55:20.939075: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-01-29 15:55:20.939093: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-01-29 15:55:20.939096: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
2025-01-29 15:55:20.939108: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-01-29 15:55:20.939115: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 7, 64, 20)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 64, 20, 1)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GroupFeatures (TimeDistributed) │ (None, 7, 960)         │        87,124 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 7, 960)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7, 64)          │        61,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GroupProbabilities (Dense)      │ (None, 7, 5)           │           325 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GroupVoting                     │ (None, 5)              │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 446,857 (1.70 MB)

 Trainable params: 148,951 (581.84 KB)

...

 Total params: 87,124 (340.33 KB)

 Trainable params: 87,122 (340.32 KB)

 Non-trainable params: 2 (8.00 B)


x shape: (9000, 7, 128, 20)
y true shape: (9000, 5)



ValueError: Input 0 of layer "Model" is incompatible with the layer: expected shape=(None, 7, 64, 20), found shape=(32, 7, 128, 20)

In [ ]:
# determiner meilleur thresholds par classes
# roc curve

from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    roc_curve
)

plt.figure(figsize=(12, 5))

best_thresholds = []
for i, specie in enumerate(train_data.species_label):
    fpr, tpr, thresholds = roc_curve(y_true[:, i], y_pred[:, i])
    gmeans = np.sqrt(tpr * (1-fpr))
    index = np.argmax(gmeans)
    best_thresholds.append( thresholds[index] )
    plt.plot(fpr, tpr, label=specie)
    plt.scatter(fpr[index], tpr[index], marker='o')

plt.title("ROC")
plt.ylabel("True Positive Rate")
plt.xlabel("False Positive Rate")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

print("Best thresholds:", best_thresholds)

In [ ]:
# precision_recall_curve

plt.figure(figsize=(12, 5))

best_thresholds = []
for i, specie in enumerate(train_data.species_label):
    precision, recall, thresholds = precision_recall_curve(y_true[:, i], y_pred[:, i])

    fscore = (2 * precision * recall) / (precision + recall)
    index = np.argmax(fscore)
    best_thresholds.append( thresholds[index] )

    plt.plot(recall, precision, label=specie)
    plt.scatter(recall[index], precision[index], marker='o')

plt.title("Recall en fonction de Precision")
plt.ylabel("Recall")
plt.xlabel("Precision")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

print("Best thresholds:", best_thresholds)
precision_recall_thresholds = np.array(best_thresholds)

In [ ]:
# fscore et thresholds a bras

def to_labels(probabilities, thresholds):
    return (probabilities >= thresholds).astype(int)

all_scores = None
all_thresholds = []

for t in np.arange(0, 1, 0.001):
    y_labels = to_labels(y_pred, t)
    scores = f1_score(y_true, y_labels, average=None)

    if all_scores is None:
        all_scores = scores
    else:
        all_scores = np.vstack((all_scores, scores))

    all_thresholds.append(t)

best_scores_index = np.argmax(all_scores, axis=0)
best_scores = all_scores[best_scores_index, range(best_scores_index.shape[0])]
best_thresholds = np.array(all_thresholds)[best_scores_index]


plt.figure(figsize=(12, 5))

for i, specie in enumerate(train_data.species_label):
    plt.plot(all_thresholds, all_scores[:, i], label=specie)
    plt.scatter(best_thresholds[i], best_scores[i])

plt.title("F1 en fonction des thresholds")
plt.ylabel("F1")
plt.xlabel("Thresholds")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

print("Best f1 scores:", best_scores)
print("Best thresholds:", best_thresholds)

In [ ]:

def analysis(dataset_type, thresholds):
    data = IndexedDataset.create(config, dataset_type)
    x = data.spectrogram_groups
    y_true = data.y
    y_pred, _ = model.predict(x, verbose=0)

    labels = deepcopy(data.species_short_label)
    labels.append("unknown")

    y_true_idx                 = np.argmax(y_true, axis=-1)
    y_true_invalid             = np.sum(y_true, axis=-1) == 0
    y_true_idx[y_true_invalid] = len(labels)

    y_pred_idx                 = np.argmax(y_pred, axis=-1)
    y_pred_invalid             = y_pred[np.arange(y_pred.shape[0]), y_pred_idx] <= thresholds[y_pred_idx]
    y_pred_idx[y_pred_invalid] = len(labels)

    confusion_matrix_analysis(str(dataset_type),
                              y_true_idx,
                              y_pred_idx,
                              target_names=labels,
                              normalize=None,
                              figsize=(5, 4),
                              xticks_rotation="vertical")

analysis(DatasetType.TRAIN, precision_recall_thresholds)